# 05 — Compare held-out TEMPTED and MEFISTO performance

This notebook creates report tables only. It does not create figures.

Generated reports are saved under `data/reports/<timestamp>/`. Figures are kept
separate and are created only by notebooks 06 and 07.


In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd

PRIMARY_METRIC = "balanced_accuracy"
root = Path(".") if Path("data").exists() else Path("..")

def newest_complete_run(method):
    runs = sorted(path for path in (root / "data" / method).iterdir() if path.is_dir())
    for run in reversed(runs):
        if (run / "all_metrics.csv").exists() and (run / "all_predictions.csv.gz").exists():
            return run
    raise FileNotFoundError(f"No completed {method} run was found")

tempted = newest_complete_run("tempted")
mefisto = newest_complete_run("mefisto")

output = root / "data" / "reports" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

metrics = pd.concat([
    pd.read_csv(tempted / "all_metrics.csv"),
    pd.read_csv(mefisto / "all_metrics.csv"),
], ignore_index=True)

predictions = pd.concat([
    pd.read_csv(tempted / "all_predictions.csv.gz"),
    pd.read_csv(mefisto / "all_predictions.csv.gz"),
], ignore_index=True)

print("TEMPTED:", tempted)
print("MEFISTO:", mefisto)


In [ ]:
metric_columns = ["accuracy", "balanced_accuracy", "macro_f1"]
summary = metrics.groupby("method")[metric_columns].agg(["count", "mean", "std", "median"]).round(4)
paired = metrics.pivot(index="batch", columns="method", values=PRIMARY_METRIC).dropna()
paired["TEMPTED_minus_MEFISTO"] = paired["TEMPTED"] - paired["MEFISTO"]
confusion = predictions.groupby(["method", "truth", "predicted"]).size().rename("count").reset_index()

metrics.to_csv(output / "all_metrics.csv", index=False)
summary.to_csv(output / "metric_summary.csv")
paired.reset_index().to_csv(output / "paired_batches.csv", index=False)
confusion.to_csv(output / "confusion_counts.csv", index=False)
(output / "comparison_report.html").write_text(
    "<h1>TEMPTED and MEFISTO comparison</h1><h2>Summary</h2>" + summary.to_html()
    + f"<h2>Paired {PRIMARY_METRIC}</h2>" + paired.to_html()
    + "<h2>All batches</h2>" + metrics.to_html(index=False), encoding="utf-8")
print("Saved:", output)
summary
